1. Реализовать программу для вывода
последовательности чисел Фибоначчи до определённого
числа в последовательности. Номер числа, до которого нужно
выводить, задаётся пользователем с клавиатуры. Для
реализации последовательности использовать генераторную
функцию.

In [10]:
from typing import Iterable


def fibonacci(n: int) -> Iterable[int]:
    previous, current = 0, 1

    for _ in range(n):
        yield previous

        previous, current = current, previous + current

fib = fibonacci(10)
for number in fib:
    print(number, end=", ")

0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 

2. Реализовать программу для бесконечной циклической
последовательности чисел (например, 1-2-3-1-2-3-1-2...).
Последовательность реализовать с помощью генераторной
функции, количество чисел для вывода задаётся
пользователем с клавиатуры.

In [12]:
def infinite_integer_number_sequence(length: int) -> Iterable[int]:
    while 1:
        for number in range(1, length+1):
            yield number

MAX_PRINT_SEQUENCE_LENGTH = 100

length = 10#int(input("Enter length > "))

inf_seq = infinite_integer_number_sequence(length)
for index, number in enumerate(inf_seq):
    print(number, end="-")

    if index >= MAX_PRINT_SEQUENCE_LENGTH:
        break

1-2-3-4-5-6-7-8-9-10-1-2-3-4-5-6-7-8-9-10-1-2-3-4-5-6-7-8-9-10-1-2-3-4-5-6-7-8-9-10-1-2-3-4-5-6-7-8-9-10-1-2-3-4-5-6-7-8-9-10-1-2-3-4-5-6-7-8-9-10-1-2-3-4-5-6-7-8-9-10-1-2-3-4-5-6-7-8-9-10-1-2-3-4-5-6-7-8-9-10-1-

3. Паттерн «Строитель» \
● Создайте класс Pizza, который содержит следующие \
атрибуты: size, cheese, pepperoni, mushrooms, onions,
bacon. \
● Создайте класс PizzaBuilder, который использует паттерн
«Строитель» для создания экземпляра Pizza. Этот класс
должен содержать методы для добавления каждого из
атрибутов Pizza. \
● Создайте класс PizzaDirector, который принимает
экземпляр PizzaBuilder и содержит метод make_pizza,
который использует PizzaBuilder для создания Pizza.

In [51]:
from abc import ABC, abstractmethod
from typing import Self
from copy import deepcopy

type Number = int | float

class Pizza:
    def __init__(self, current_pizza: Self = None) -> None:
        self._size: Number | None = None
        self._cheese: bool = False
        self._pepperoni: bool = False
        self._mushrooms: bool = False
        self._onions: bool = False
        self._bacon: bool = False

        self._aliases = {
            "cheese": self._cheese,
            "pepperoni": self._pepperoni,
            "mushrooms": self._mushrooms,
            "onions": self._onions,
            "bacon": self._bacon,
        }

        if current_pizza is not None:
            if not isinstance(current_pizza, Pizza):
                raise TypeError("current_pizza must be of type Pizza or None")

            self._size = current_pizza.size
            for ingredient in current_pizza.ingredients:
                self._aliases[ingredient] = True


    @property
    def possible_ingredients(self) -> Iterable[str]:
        return self._aliases.keys()

    @property
    def size(self) -> Number:
        return self._size

    @size.setter
    def size(self, value: Number) -> None:
        if not isinstance(value, Number.__value__):
            raise TypeError(f"size must be a number: {Number.__value__}")

        if value < 0:
            raise ValueError("size must be >= 0")

        self._size = value

    def add_product(self, name: str) -> None:
        if name not in self._aliases:
            raise ValueError(f"'{name}' is not a valid product name")

        self._aliases[name] = True

    @property
    def ingredients(self) -> Iterable[str]:
        for ingredient in self._aliases:
            if self._aliases[ingredient]:
                yield ingredient

    def __str__(self) -> str:
        if not self._size:
            return "Infinite small pizza"

        pizza_description = f"Pizza with size: {self._size}."

        contains_products = tuple(ingredient for ingredient in self._aliases if self._aliases[ingredient])

        if contains_products:
            pizza_description += f" Contains products: {', '.join(contains_products)}."

        return pizza_description


class PizzaBuilderAbstract(ABC):
    @abstractmethod
    def set_size(self, size: Number) -> Self: ...

    @abstractmethod
    def add_cheese(self) -> Self: ...

    @abstractmethod
    def add_pepperoni(self) -> Self: ...

    @abstractmethod
    def add_mushrooms(self) -> Self: ...

    @abstractmethod
    def add_onions(self) -> Self: ...

    @abstractmethod
    def add_bacon(self) -> Self: ...


class PizzaBuilder(PizzaBuilderAbstract):
    def __init__(self, pizza: Pizza = Pizza()) -> None:
        if not isinstance(pizza, Pizza):
            raise TypeError("pizza must be of type Pizza")

        self._pizza = pizza

    def get_list_ingredients(self) -> Iterable[str]:
        return self._pizza.possible_ingredients

    def set_size(self, size: Number) -> Self:
        if not isinstance(size, Number.__value__):
            raise TypeError(f"size must be a number: {Number.__value__}")

        self._pizza.size = size
        return self

    def add_cheese(self) -> Self:
        self._pizza.add_product("cheese")
        return self

    def add_pepperoni(self) -> Self:
        self._pizza.add_product("pepperoni")
        return self

    def add_mushrooms(self) -> Self:
        self._pizza.add_product("mushrooms")
        return self

    def add_onions(self) -> Self:
        self._pizza.add_product("onions")
        return self

    def add_bacon(self) -> Self:
        self._pizza.add_product("bacon")
        return self

    def get_built_pizza(self) -> Pizza:
        return Pizza(self._pizza)

        # OR return deepcopy(self._pizza)

class Director:
    def __init__(self, builder: PizzaBuilder = PizzaBuilder()) -> None:
        if not isinstance(builder, PizzaBuilder):
            raise TypeError("builder must be of type PizzaBuilder")

        self._builder = builder

    @property
    def builder(self) -> PizzaBuilder:
        return self._builder

    @builder.setter
    def builder(self, builder: PizzaBuilder) -> None:
        if not isinstance(builder, PizzaBuilder):
            raise TypeError("builder must be of type PizzaBuilder")

        self._builder = builder

    def help(self):
        possible_ingredients = ", ".join(self._builder.get_list_ingredients())

        help_text = f"""To create pizza with size X and ingredients ({possible_ingredients}...) do this:
            <director>.make_pizza_with_<ingredient1>_<ingredient2>_...(<size>)
        """

        print(help_text)
        return help_text

    def __getattribute__(self, item):
        if item.startswith("make_pizza_with"):
            ingredients = item.split("_")[3:]

            for ingredient in ingredients:
                try:
                    self._builder.__getattribute__("add_" + ingredient)()
                except AttributeError as e:
                    raise ExceptionGroup("Ingredients exceptions:", (
                        e,
                        AttributeError(f"Can't add {ingredient} (invalid) to pizza. Please view <director>.help()")
                    ))

            return lambda size: self._builder.set_size(size).get_built_pizza()


        return super().__getattribute__(item)

director = Director()
print(director.make_pizza_with_bacon_cheese(47))


Pizza with size: 47. Contains products: cheese, bacon.


In [52]:
# Тест 1: пустая пицца
pizza = Pizza()
assert pizza.size is None, "Пустая пицца должна иметь size=None"
assert list(pizza.ingredients) == [], "Пустая пицца не должна содержать ингредиенты"

# Тест 2: установка размера и добавление ингредиентов
builder = PizzaBuilder()
pizza = builder.set_size(30).add_cheese().add_bacon().get_built_pizza()
assert pizza.size == 30, "Размер пиццы должен быть 30"
assert "cheese" in pizza.ingredients, "Пицца должна содержать сыр"
assert "bacon" in pizza.ingredients, "Пицца должна содержать бекон"
assert "onions" not in pizza.ingredients, "Пицца не должна содержать лук"

# Тест 3: проверка типа размера (int и float)
pizza_int = PizzaBuilder().set_size(25).get_built_pizza()
pizza_float = PizzaBuilder().set_size(33.5).get_built_pizza()
assert pizza_int.size == 25, "Размер должен быть 25 (int)"
assert pizza_float.size == 33.5, "Размер должен быть 33.5 (float)"

# Тест 4: недопустимый тип размера (строка)
try:
    PizzaBuilder().set_size("large")
    assert False, "Должно быть исключение при передаче строки в set_size"
except TypeError:
    pass

# Тест 5: установка отрицательного размера
try:
    p = Pizza()
    p.size = -5
    assert False, "Должно быть исключение при отрицательном размере"
except ValueError:
    pass

# Тест 6: попытка добавить несуществующий ингредиент
try:
    pizza = Pizza()
    pizza.add_product("broccoli")
    assert False, "Должно быть исключение при добавлении несуществующего ингредиента"
except ValueError:
    pass

# Тест 7: создание пиццы через Director
director = Director()
pizza = director.make_pizza_with_cheese_mushrooms_onions(42)
assert pizza.size == 42, "Размер должен быть 42"
assert "cheese" in pizza.ingredients, "Пицца должна содержать сыр"
assert "mushrooms" in pizza.ingredients, "Пицца должна содержать грибы"
assert "onions" in pizza.ingredients, "Пицца должна содержать лук"

# Тест 8: вызов help()
help_text = director.help()
assert "cheese" in help_text and "pepperoni" in help_text, "Help должен показывать ингредиенты"

# Тест 9: неверный ингредиент через Director
error_thrown = False
try:
    director.make_pizza_with_chocolate(50)
except ExceptionGroup as e:
    error_thrown = True
    assert any("Can't add chocolate" in str(err) for err in e.exceptions), "Ошибка должна содержать chocolate"
assert error_thrown, "Должна быть вызвана ошибка при неправильном ингредиенте"

print("✅ Все тесты пройдены успешно!")


To create pizza with size X and ingredients (cheese, pepperoni, mushrooms, onions, bacon...) do this:
            <director>.make_pizza_with_<ingredient1>_<ingredient2>_...(<size>)
        
✅ Все тесты пройдены успешно!


4. Паттерн «Фабричный метод» \
● Создайте абстрактный класс Animal, у которого есть
абстрактный метод speak. \
● Создайте классы Dog и Cat, которые наследуют от Animal
и реализуют метод speak. \
● Создайте класс AnimalFactory, который использует
паттерн «Фабричный метод» для создания экземпляра
Animal. Этот класс должен иметь метод create_animal,
который принимает строку («dog» или «cat») и возвращает
соответствующий объект (Dog или Cat).

In [54]:
from abc import ABC, abstractmethod

class Animal(ABC):
    @abstractmethod
    def speak(self) -> None: ...


class Dog(Animal):
    def speak(self) -> None:
        print("Dodododoodo")

class Cat(Animal):
    def speak(self) -> None:
        print("Meow")


class AnimalFactory:
    @staticmethod
    def create_animal(type_: str) -> Animal:
        if type_ == "dog":
            return Dog()
        elif type_ == "cat":
            return Cat()

        raise ValueError("Invalid animal type provided.")



In [55]:
def test_create_dog():
    animal = AnimalFactory.create_animal("dog")
    assert isinstance(animal, Dog), "Dog creation failed"
    assert isinstance(animal, Animal), "Dog should be an instance of Animal"
    animal.speak()  # должно напечатать: Dodododoodo

def test_create_cat():
    animal = AnimalFactory.create_animal("cat")
    assert isinstance(animal, Cat), "Cat creation failed"
    assert isinstance(animal, Animal), "Cat should be an instance of Animal"
    animal.speak()  # должно напечатать: Meow

def test_invalid_type():
    try:
        AnimalFactory.create_animal("dragon")
        assert False, "Should have raised ValueError for invalid animal type"
    except ValueError as e:
        print("Caught expected exception:", e)

def test_type_enforcement():
    # Проверка, что объекты Dog и Cat действительно реализуют метод speak
    dog = Dog()
    cat = Cat()
    assert hasattr(dog, "speak") and callable(getattr(dog, "speak")), "Dog missing speak()"
    assert hasattr(cat, "speak") and callable(getattr(cat, "speak")), "Cat missing speak()"

# Запуск тестов
test_create_dog()
test_create_cat()
test_invalid_type()
test_type_enforcement()
print("All tests passed.")


Dodododoodo
Meow
Caught expected exception: Invalid animal type provided.
All tests passed.


5. Паттерн «Стратегия» \
● Создайте класс Calculator, который использует разные
стратегии для выполнения математических операций. \
● Создайте несколько классов, каждый реализует
определенную стратегию математической операции,
например, Addition, Subtraction, Multiplication, Division.
Каждый класс должен содержать метод execute, который
принимает два числа и выполняет соответствующую
операцию. \
● Calculator должен иметь метод set_strategy, который
устанавливает текущую стратегию, и метод calculate,
который выполняет операцию с помощью текущей стратегии.

In [77]:
def verify_values_type_is_number(*variables_values: Number) -> None:
    if not all(map(lambda value: isinstance(value, Number.__value__), variables_values)):
        raise TypeError("Values should be numbers.")

class Strategy(ABC):
    @abstractmethod
    def calculate(self, *args: Number) -> Number: ...

class AdditionStrategy(Strategy):
    def calculate(self, *args: Number) -> Number:
        verify_values_type_is_number(*args)
        return sum(args)

class SubtractionStrategy(Strategy):
    def calculate(self, *args: Number) -> Number:
        verify_values_type_is_number(*args)

        result = 0

        for number in args:
            result -= number

        return result

class MultiplicationStrategy(Strategy):
    def calculate(self, *args: Number) -> Number:
        verify_values_type_is_number(*args)

        args = tuple(args)

        if len(args) < 1:
            raise ValueError("args must contain at least one number")
        elif len(args) == 1:
            return args[0]

        result = args[0]
        for number in args[1:]:
            result *= number

        return result

class DivisionStrategy(Strategy):
    def calculate(self, *args: Iterable[Number]) -> Number:
        verify_values_type_is_number(*args)

        args = tuple(args)

        if len(args) < 1:
            raise ValueError("args must contain at least one number")
        elif len(args) == 1:
            return args[0]

        result = args[0]
        for number in args[1:]:
            result /= number

        return result


class Calculator:
    def __init__(self, strategy: Strategy, initial_number: Number | None = None) -> None:
        if not isinstance(strategy, Strategy):
            raise TypeError("Strategy should be an instance of Strategy")

        self._strategy: Strategy = strategy
        self._result: Number | None = initial_number

    def set_strategy(self, strategy: Strategy):
        if not isinstance(strategy, Strategy):
            raise TypeError("Strategy should be an instance of Strategy")

        self._strategy = strategy

    def calculate(self, *args: Number) -> Self:
        if self._result is None:
            self._result = self._strategy.calculate(*args)
        else:
            self._result = self._strategy.calculate(self._result, *args)

        return self

    @property
    def calculation_result(self):
        return self._result


In [79]:
def test_addition():
    calc = Calculator(AdditionStrategy(), 10)
    calc.calculate(5, 3)
    assert calc.calculation_result == 18, "Addition failed"

def test_subtraction():
    calc = Calculator(SubtractionStrategy(), 20)
    calc.calculate(5, 3)
    # 0 - 5 - 3 - 20 = -28
    assert calc.calculation_result == -28, "Subtraction failed"

def test_multiplication():
    calc = Calculator(MultiplicationStrategy(), 2)
    calc.calculate(3, 4)
    # 3 * 4 * 2 = 24
    assert calc.calculation_result == 24, "Multiplication failed"

def test_division():
    calculator = Calculator(DivisionStrategy(), 100)

    calculator.calculate(2)  # 100 / 2 = 50
    assert abs(calculator.calculation_result - 50) < 1e-6, "Division by 2 failed"

    calculator.calculate(5)  # 50 / 5 = 10
    assert abs(calculator.calculation_result - 10) < 1e-6, "Division by 5 failed"

    calculator.calculate(2, 2)  # 10 / 2 / 2 = 2.5
    assert abs(calculator.calculation_result - 2.5) < 1e-6, "Multiple division failed"

    try:
        Calculator(DivisionStrategy()).calculate()  # пустые args — ValueError
    except ValueError:
        pass
    else:
        assert False, "Empty args should raise ValueError"

    try:
        Calculator(DivisionStrategy()).calculate(10, 0)  # деление на ноль
    except ZeroDivisionError:
        pass
    else:
        assert False, "Division by zero should raise ZeroDivisionError"


def test_strategy_switching():
    calc = Calculator(AdditionStrategy(), 10)
    calc.calculate(5)
    assert calc.calculation_result == 15, "Addition failed"

    calc.set_strategy(MultiplicationStrategy())
    calc.calculate(2)
    assert calc.calculation_result == 30, "Multiplication after switch failed"

def test_type_checking():
    try:
        calc = Calculator(AdditionStrategy(), 0)
        calc.calculate(5, "wrong")
        assert False, "TypeError expected for non-numeric input"
    except TypeError:
        print("Caught expected TypeError for input values")

def test_strategy_type_checking():
    try:
        Calculator("not-a-strategy", 0)
        assert False, "TypeError expected for invalid strategy"
    except TypeError:
        print("Caught expected TypeError for invalid strategy")

def test_multiple_strategy_switching():
    calc = Calculator(AdditionStrategy(), 0)
    calc.calculate(5, 10)  # 5 + 10 + 0 = 15
    assert calc.calculation_result == 15, "Addition failed in multiple strategy switching"

    calc.set_strategy(SubtractionStrategy())
    calc.calculate(5)  # 0 - 5 - 15 = -20
    assert calc.calculation_result == -20, "Subtraction failed in multiple strategy switching"

    calc.set_strategy(MultiplicationStrategy())
    calc.calculate(2)  # 2 * -20 = -40
    assert calc.calculation_result == -40, "Multiplication failed in multiple strategy switching"

    calc.set_strategy(AdditionStrategy())
    calc.calculate(100)  # 100 + (-40) = 60
    assert calc.calculation_result == 60, "Final addition failed in multiple strategy switching"

    print("Multiple strategy switching test passed.")


# Повторный запуск всех тестов, включая новый:
test_addition()
test_subtraction()
test_multiplication()
test_division()
test_strategy_switching()
test_type_checking()
test_strategy_type_checking()
test_multiple_strategy_switching()

print("All tests passed successfully.")


Caught expected TypeError for input values
Caught expected TypeError for invalid strategy
Multiple strategy switching test passed.
All tests passed successfully.
